# LLM evaluation pipeline (Colab)

**Runtime → Change runtime type → GPU** (T4+ recommended).

Runs: **vLLM** → **lm-eval** → **load test** → **guardrails** → **improve** (HellaSwag subset + McNemar stats).

**Bring your code:** by default the next cell clones **[atulsaini04/llm-eval-pipeline](https://github.com/atulsaini04/llm-eval-pipeline)**. To use a fork or zip upload instead, set `GITHUB_REPO_URL = None` and upload `assign.zip` when prompted (must contain `serve/`, `eval_runner/`, …).

In [ ]:
# --- Configuration (edit these) ---
GITHUB_REPO_URL = "https://github.com/atulsaini04/llm-eval-pipeline.git"  # None = upload zip instead
MODEL = "HuggingFaceTB/SmolLM3-3B"
COLAB_ASSIGN_ROOT = "/content/assign"
VLLM_PORT = "8000"
EVAL_LIMIT = 0.02
SUBSET_N = 100
VOTE_K = 5
EVAL_TASKS = "hellaswag,mmlu_stem,custom_json_qa"
# If Step B fails, try: "hellaswag,custom_json_qa" first (mmlu_stem is many subtasks + slow).
# vLLM memory (raise if you have A100; lower further if OOM on T4)
VLLM_MAX_MODEL_LEN = 2048
VLLM_GPU_MEM_UTIL = 0.78
VLLM_MAX_NUM_SEQS = 32
# Part C: use /v1/completions (same as lm-eval); --quick = smaller matrix
LOAD_TEST_QUICK = True
LOAD_TEST_CONCURRENCY = "1 4"


In [ ]:
import os, shutil, subprocess, sys, zipfile
from pathlib import Path

def has_gpu():
    try:
        subprocess.run(
            ["nvidia-smi"], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
        )
        return True
    except Exception:
        return False

print("nvidia-smi OK:", has_gpu())
if not has_gpu():
    raise SystemExit("Enable GPU: Runtime → Change runtime type → GPU")

def find_repo_root(base: Path):
    for p in base.rglob("serve.py"):
        if p.parent.name == "serve":
            return p.parent.parent
    return None

def ensure_repo():
    root = Path(COLAB_ASSIGN_ROOT)
    if (root / "serve" / "serve.py").is_file():
        print("Repo already at", root)
        return root
    if GITHUB_REPO_URL:
        if root.exists():
            shutil.rmtree(root)
        parent = root.parent
        subprocess.check_call(["git", "clone", GITHUB_REPO_URL, str(root)], cwd=str(parent))
        return root
    from google.colab import files
    print("Upload assign.zip (repo root with serve/, eval_runner/, …)")
    up = files.upload()
    assert up, "No upload"
    zpath = Path("/content/_up.zip")
    zpath.write_bytes(list(up.values())[0])
    ex = Path("/content/_extract")
    if ex.exists():
        shutil.rmtree(ex)
    ex.mkdir(parents=True)
    with zipfile.ZipFile(zpath, "r") as z:
        z.extractall(ex)
    found = find_repo_root(ex)
    if found is None:
        raise RuntimeError("Could not find serve/serve.py in zip")
    if root.exists():
        shutil.rmtree(root)
    shutil.move(str(found), str(root))
    print("Repo ready at", root)
    return root

ROOT = ensure_repo()
os.environ["COLAB_ASSIGN_ROOT"] = str(ROOT)
os.environ["MODEL"] = MODEL
os.environ["VLLM_PORT"] = VLLM_PORT
os.environ["OPENAI_BASE_URL"] = f"http://127.0.0.1:{VLLM_PORT}/v1"
os.environ["OPENAI_API_KEY"] = "EMPTY"
os.environ["VLLM_COMPLETIONS_URL"] = f"http://127.0.0.1:{VLLM_PORT}/v1/completions"
os.environ["HF_HOME"] = "/content/hf"
print("ROOT=", ROOT)


## Install dependencies
`(pip + vLLM can take several minutes)`

In [ ]:
import os, subprocess, sys
ROOT = os.environ["COLAB_ASSIGN_ROOT"]
os.chdir(ROOT)
os.makedirs(os.environ.get("HF_HOME", "/content/hf"), exist_ok=True)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "pip"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "vllm"])
print("pip install done.")


## Part A — Start vLLM (background)

In [ ]:
import os, subprocess, time, requests
ROOT = os.environ["COLAB_ASSIGN_ROOT"]
MODEL = os.environ["MODEL"]
port = os.environ["VLLM_PORT"]
log = open("/tmp/vllm_colab.log", "w")
subprocess.run("pkill -f 'vllm serve' || true", shell=True)
time.sleep(2)
# Colab T4 (~15GB): avoid OOM during sampler warmup (256 dummy seqs in some vLLM versions).
cmd = [
    "vllm", "serve", MODEL, "--host", "0.0.0.0", "--port", port,
    "--dtype", "half",
    "--max-model-len", str(VLLM_MAX_MODEL_LEN),
    "--gpu-memory-utilization", str(VLLM_GPU_MEM_UTIL),
    "--max-num-seqs", str(VLLM_MAX_NUM_SEQS),
]
print("Starting:", " ".join(cmd))
subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, cwd=ROOT, start_new_session=True)
base = f"http://127.0.0.1:{port}"
ready = False
for i in range(90):
    try:
        r = requests.get(f"{base}/v1/models", timeout=5)
        if r.ok:
            ready = True
            print("vLLM ready:", str(r.json())[:500])
            break
    except Exception as e:
        if i % 10 == 0:
            print("waiting...", i, e)
    time.sleep(5)
if not ready:
    print(open("/tmp/vllm_colab.log").read()[-6000:])
    raise RuntimeError("vLLM did not become ready")


## Part B — lm-eval

In [ ]:
import os, subprocess, sys
ROOT = os.environ["COLAB_ASSIGN_ROOT"]
cmd = [
    sys.executable, "eval_runner/run_eval.py",
    "--model", os.environ["MODEL"],
    "--base-url", os.environ["VLLM_COMPLETIONS_URL"],
    "--tasks", EVAL_TASKS,
    "--limit", str(EVAL_LIMIT),
    "--num-concurrent", "1",
    "--max-length", str(VLLM_MAX_MODEL_LEN),
    "--output", "results/eval_results.json",
    "--summary", "results/summary.md",
]
print("RUN:", " ".join(cmd))
p = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True)
print(p.stdout)
if p.stderr:
    print("STDERR:\n", p.stderr)
if p.returncode != 0:
    raise RuntimeError(f"run_eval failed ({p.returncode}). See STDERR above.")
print(open(os.path.join(ROOT, "results/summary.md"), encoding="utf-8").read()[:4000])


## Part C — Load test

In [ ]:
import os, subprocess, sys
ROOT = os.environ["COLAB_ASSIGN_ROOT"]
cmd = [
    sys.executable, "perf/load_test.py",
    "--model", os.environ["MODEL"],
    "--base-url", os.environ["OPENAI_BASE_URL"],
    "--output", "metrics.csv",
    "--completions",
    "--overwrite",
] + (["--quick"] if LOAD_TEST_QUICK else []) + [
    "--concurrency",
] + LOAD_TEST_CONCURRENCY.split()
print("RUN:", " ".join(cmd))
p = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True)
print(p.stdout)
if p.stderr:
    print("STDERR:\n", p.stderr)
if p.returncode != 0:
    raise RuntimeError(f"load_test failed ({p.returncode}). See STDERR above.")
print(open(os.path.join(ROOT, "metrics.csv"), encoding="utf-8").read()[:2500])


## Part D — Guardrails

In [ ]:
import os, subprocess, sys
ROOT = os.environ["COLAB_ASSIGN_ROOT"]
subprocess.check_call([sys.executable, "guardrails/validate.py", "--model", os.environ["MODEL"]], cwd=ROOT)


## Part E — Improve pipeline

In [ ]:
import os, subprocess, sys
ROOT = os.environ["COLAB_ASSIGN_ROOT"]
env = os.environ.copy()
env["PYTHONPATH"] = ROOT + os.pathsep + env.get("PYTHONPATH", "")
env["SUBSET_N"] = str(SUBSET_N)
env["VOTE_K"] = str(VOTE_K)
subprocess.check_call(["bash", "improve/eval.sh"], cwd=ROOT, env=env)
print(open(os.path.join(ROOT, "improve/runs/stats.json"), encoding="utf-8").read())


## Download results zip

In [ ]:
import os, shutil
from pathlib import Path
from google.colab import files
ROOT = Path(os.environ["COLAB_ASSIGN_ROOT"])
base = "/content/assign_results"
if Path(base + ".zip").exists():
    os.remove(base + ".zip")
shutil.make_archive(base, "zip", root_dir=str(ROOT))
files.download(base + ".zip")
print("Downloaded assign_results.zip")
